In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt

# 1. Cargar datos
df = pd.read_csv('polymarket_pro_dataset_with_resolution.csv', sep=';')
df['timestamp'] = pd.to_datetime(df['timestamp'])

# ORDEN CRÍTICO: Para evitar Target Leakage y calcular series temporales correctamente
df = df.sort_values(by=['market_slug', 'timestamp']).reset_index(drop=True)

# ==========================================
# 3. FEATURE ENGINEERING (Microestructura y L3)
# ==========================================

# A. Volúmenes Agregados (Top 3 Niveles)
for side in ['up', 'down']:
    # Suma de volumen de Bid y Ask en los 3 niveles
    df[f'{side}_total_bid_vol'] = df[f'{side}_bid_s_1'] + df[f'{side}_bid_s_2'] + df[f'{side}_bid_s_3']
    df[f'{side}_total_ask_vol'] = df[f'{side}_ask_s_1'] + df[f'{side}_ask_s_2'] + df[f'{side}_ask_s_3']
    
    # B. Order Book Imbalance Global (L3 OBI)
    # Rango de [-1, 1]. -1 = Presión total de venta, 1 = Presión total de compra
    df[f'{side}_obi_l3'] = (df[f'{side}_total_bid_vol'] - df[f'{side}_total_ask_vol']) / \
                           (df[f'{side}_total_bid_vol'] + df[f'{side}_total_ask_vol'] + 1e-8)
    
    # C. Weighted Mid Price (WMP) del Nivel 1
    # Pondera el precio por el volumen contrario. Más preciso que el Mid Price simple.
    df[f'{side}_wmp_1'] = (df[f'{side}_bid_p_1'] * df[f'{side}_ask_s_1'] + df[f'{side}_ask_p_1'] * df[f'{side}_bid_s_1']) / \
                          (df[f'{side}_bid_s_1'] + df[f'{side}_ask_s_1'] + 1e-8)
                          
    # Spread del Nivel 1
    df[f'{side}_spread_1'] = df[f'{side}_ask_p_1'] - df[f'{side}_bid_p_1']

# ==========================================
# 4. FEATURE ENGINEERING (Dinámica de BTC)
# ==========================================

# Asumiendo que cada fila es aprox 10 segundos:
# Span 6 = ~1 minuto; Span 18 = ~3 minutos
df['btc_ema_1m'] = df['btc_spot'].transform(lambda x: x.ewm(span=6, adjust=False).mean())
df['btc_ema_3m'] = df['btc_spot'].transform(lambda x: x.ewm(span=18, adjust=False).mean())

# D. Distancia del precio actual a la EMA (Oscilador de tendencia)
# Valores positivos indican que BTC está por encima de su tendencia reciente (Bullish momentum)
df['btc_dist_ema_1m'] = (df['btc_spot'] / df['btc_ema_1m']) - 1
df['btc_dist_ema_3m'] = (df['btc_spot'] / df['btc_ema_3m']) - 1

# Retorno porcentual suavizado (ventana de 6 periodos = 1 minuto atrás)
# Esto evita el ruido del segundo a segundo
df['btc_return_1m'] = df.groupby('market_slug')['btc_spot'].pct_change(periods=6)

# Limpieza de nulos generados por los cálculos temporales (retornos y EMAs iniciales)
#df = df.dropna().reset_index(drop=True)

In [ ]:
# ==========================================
# 5. PREPARACIÓN DEL MODELO
# ==========================================

# Variable objetivo
df['target'] = df['resolution'].map({'up': 1, 'down': 0})

# Seleccionamos las variables finales
features = [
    'seconds_left', 'btc_spot',
    'btc_dist_ema_1m', 'btc_dist_ema_3m', 'btc_return_1m',
    'up_wmp_1', 'down_wmp_1',
    'up_spread_1', 'down_spread_1',
    'up_obi_l3', 'down_obi_l3',
    'up_total_bid_vol', 'up_total_ask_vol',
    'down_total_bid_vol', 'down_total_ask_vol'
]

X = df[features]
y = df['target']

# Split temporal (Simulación de entorno real)
# En lugar de un split aleatorio, cogemos el primer 80% del tiempo para entrenar 
# y el último 20% para validar. Esto previene completamente el target leakage temporal.
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# ==========================================
# 6. ENTRENAMIENTO XGBOOST
# ==========================================

model = xgb.XGBClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8, # Ayuda a prevenir sobreajuste
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

print("Entrenando XGBoost...")
model.fit(X_train, y_train)

# Evaluación
y_pred = model.predict(X_test)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred, target_names=['Down (0)', 'Up (1)']))

# Importancia de variables
xgb.plot_importance(model, importance_type='gain', max_num_features=15)
plt.title("Importancia de Features (Gain)")
plt.show()

0       0.000095
1       0.000095
2       0.000238
3       0.000295
4       0.000209
          ...   
6122    0.000129
6123   -0.000136
6124   -0.000227
6125   -0.000617
6126   -0.000461
Name: btc_return_1m, Length: 6127, dtype: float64